In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

## Reading Data Using AUTOLOADER

In [0]:
%sql
-- Preview the data
SELECT COUNT(*)
FROM parquet.`abfss://source@storageaccountsunnybest.dfs.core.windows.net/orders`

## Parameterization

In [0]:
dbutils.widgets.text("filename", "")

In [0]:
file_name = dbutils.widgets.get("filename")

In [0]:
file_name

In [0]:
location = f'abfss://bronze@storageaccountsunnybest.dfs.core.windows.net/checkpoint_{file_name}'
cloud_files_path = f'abfss://source@storageaccountsunnybest.dfs.core.windows.net/{file_name}'

order_df = spark.readStream.format("cloudFiles")\
    .option("cloudFiles.format", "parquet")\
    .option("cloudFiles.schemaLocation", location)\
    .option("cloudFiles.schemaEvolutionMode", "addNewColumns")\
    .load(cloud_files_path)

## Data Writing

In [0]:
order_df.writeStream.format("parquet")\
    .outputMode("append")\
    .option("checkpointLocation", location)\
    .option('path', f'abfss://bronze@storageaccountsunnybest.dfs.core.windows.net/{file_name}')\
    .trigger(availableNow=True)\
    .start()

In [0]:
df = spark.read.parquet(f'abfss://bronze@storageaccountsunnybest.dfs.core.windows.net/{file_name}')
display(df)